In [1]:
%pip install datasets pandas

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from datasets import load_dataset
import pandas as pd

# 从hugging face远程拉取Spider数据集切片
dataset = load_dataset("xlangai/spider", split="train")

# 将HF的Dataset对象直接转化为 Pandas DataFrame
df_raw=dataset.to_pandas()
print(f"数据拉取成功，总量: {len(df_raw)} 条")

README.md:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

spider/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  831kB            

spider/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

spider/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  126kB            

spider/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

数据拉取成功，总量: 7000 条


In [5]:
df_raw.head()

,db_id,query,question,query_toks,query_toks_no_value,question_toks
0,department_management,SELECT count(*) FROM head WHERE age > 56,How many heads of the departments are older th...,"[SELECT, count, (, *, ), FROM, head, WHERE, ag...","[select, count, (, *, ), from, head, where, ag...","[How, many, heads, of, the, departments, are, ..."
1,department_management,"SELECT name , born_state , age FROM head ORD...","List the name, born state and age of the heads...","[SELECT, name, ,, born_state, ,, age, FROM, he...","[select, name, ,, born_state, ,, age, from, he...","[List, the, name, ,, born, state, and, age, of..."
2,department_management,"SELECT creation , name , budget_in_billions ...","List the creation year, name and budget of eac...","[SELECT, creation, ,, name, ,, budget_in_billi...","[select, creation, ,, name, ,, budget_in_billi...","[List, the, creation, year, ,, name, and, budg..."
3,department_management,"SELECT max(budget_in_billions) , min(budget_i...",What are the maximum and minimum budget of the...,"[SELECT, max, (, budget_in_billions, ), ,, min...","[select, max, (, budget_in_billions, ), ,, min...","[What, are, the, maximum, and, minimum, budget..."
4,department_management,SELECT avg(num_employees) FROM department WHER...,What is the average number of employees of the...,"[SELECT, avg, (, num_employees, ), FROM, depar...","[select, avg, (, num_employees, ), from, depar...","[What, is, the, average, number, of, employees..."


In [10]:
print(df_raw[['db_id','question','query']].head())

                   db_id                                           question  \
0  department_management  How many heads of the departments are older th...   
1  department_management  List the name, born state and age of the heads...   
2  department_management  List the creation year, name and budget of eac...   
3  department_management  What are the maximum and minimum budget of the...   
4  department_management  What is the average number of employees of the...   

                                               query  
0         SELECT count(*) FROM head WHERE age  >  56  
1  SELECT name ,  born_state ,  age FROM head ORD...  
2  SELECT creation ,  name ,  budget_in_billions ...  
3  SELECT max(budget_in_billions) ,  min(budget_i...  
4  SELECT avg(num_employees) FROM department WHER...  


In [11]:
#数据清洗

df_cleaned=df_raw[['db_id','question','query']].copy()



In [13]:
# 查找清洗后的数据中有没有空值或者空行,如果有则进行删除包含空值和空行的行
df_cleaned.isnull().sum()
print(df_cleaned.isnull().sum())
df_cleaned.dropna(inplace=True)
print(df_cleaned.isnull().sum())

db_id       0
question    0
query       0
dtype: int64
db_id       0
question    0
query       0
dtype: int64


In [16]:
#查找问题长度低于10的行，太简短的问题没必要保留
df_cleaned = df_cleaned[df_cleaned['question'].str.len() > 10]
print(df_cleaned['question'].str.len().describe())

count    7000.000000
mean       70.864857
std        23.890461
min        16.000000
25%        54.000000
50%        69.000000
75%        86.000000
max       224.000000
Name: question, dtype: float64


In [18]:
#查找query中是否有出现特定乱码或者错误字符的行（假设发现有error）
df_cleaned[~df_cleaned['query'].str.contains('ERROR', na=False)]

#清洗后剩余的数据量
print(f"清洗后剩余数据量: {len(df_cleaned)} 条")


清洗后剩余数据量: 7000 条


In [44]:
import json
# 构建指令格式


formatted_data=[]

# itertuples() 是 Pandas 中遍历数据最高效的方法之一
for row in df_cleaned.itertuples(index=False):
    # 构建Instruction：告诉模型的任务是什么
    instruction = "You are a senior SQL developer. Write a SQL query that answers the user's question based on the database schema."

    # 构建 Input: 用户的实际问题,以及属于哪种数据库
    input_text = f'Database:{row.db_id}\nQuestion:{row.question}'

    # 构建Output: 模型需要生成纯正的SQL
    output_text = row.query

    # 组装成标准字典
    formatted_data.append({
        "instruction": instruction,
        "input": input_text,
        "output": output_text
    })

    # 我们只取前 3000 条高质量数据进行快速微调实操
final_dataset = formatted_data[:3000]

print(final_dataset[0])
print("instruction:", final_dataset[0]["instruction"])
print("input:", final_dataset[0]["input"])
print("output:", final_dataset[0]["output"])

print(f"已生成标准格式数据 {len(final_dataset)} 条")

output_path = "/content/drive/MyDrive/finetune_text_to_sql.jsonl"
with open(output_path, "w", encoding="utf-8") as f:
    for item in final_dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"清洗后的数据已安全保存至 Google Drive: {output_path}")

{'instruction': "You are a senior SQL developer. Write a SQL query that answers the user's question based on the database schema.", 'input': 'Database:department_management\nQuestion:How many heads of the departments are older than 56 ?', 'output': 'SELECT count(*) FROM head WHERE age  >  56'}
instruction: You are a senior SQL developer. Write a SQL query that answers the user's question based on the database schema.
input: Database:department_management
Question:How many heads of the departments are older than 56 ?
output: SELECT count(*) FROM head WHERE age  >  56
已生成标准格式数据 3000 条
清洗后的数据已安全保存至 Google Drive: /content/drive/MyDrive/finetune_text_to_sql.jsonl


In [23]:
%pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 64.7 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 155.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [24]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [29]:
max_seq_length = 2048
dtype=torch.bfloat16
load_in_4bit = False

model,tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print('基础设置完成')

==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct as a legacy tokenizer.


基础设置完成


In [30]:
# 注入LoRA配置
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "v_proj",
        "k_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_gradient_checkpointing="unsloth",
)
print('LoRA配置已注入')

Unsloth 2026.9.11 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


LoRA配置已注入


In [48]:

from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
### Instruction:
{}
### Input:
{}
### Response:
{}
"""



EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs=examples["output"]
    
    texts=[]
    
    for instruction, input, output in zip(instructions, inputs, outputs):
        text=prompt.format(instruction, input, output)+EOS_TOKEN
        texts.append(text)
        
    return {"text": texts}




In [49]:
dataset=load_dataset("json", data_files="/content/drive/MyDrive/finetune_text_to_sql.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [58]:
# 设置训练器开始训练
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        # max_steps=60,
        num_train_epochs=3,
        warmup_steps=10,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_torch_fused",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="output",
        report_to="none",
        logging_steps=10,  
    ),
)

In [59]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 3 | Total steps = 282
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.301615
20,0.289730
30,0.290541
40,0.298524
50,0.306641
60,0.301496
70,0.320797
80,0.294504
90,0.273394
100,0.232923


Unsloth: Restored added_tokens_decoder metadata in output/checkpoint-282/tokenizer_config.json.


In [52]:
print(trainer_stats)
print(model.num_parameters(only_trainable=True))

TrainOutput(global_step=60, training_loss=0.4191205064455668, metrics={'train_runtime': 94.6447, 'train_samples_per_second': 20.286, 'train_steps_per_second': 0.634, 'total_flos': 1.3265219939794944e+16, 'train_loss': 0.4191205064455668, 'epoch': 0.64})
41943040


In [60]:
# 开启推理模式，提升运行速度
FastLanguageModel.for_inference(model)

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
### Instruction:
{}
### Input:
{}
### Response:
{}
"""

# 准备测试数据字典
test_cases = [
    {
        "instruction": "You are a senior SQL developer. Write a SQL query that answers the user's question based on the database schema.",
        "input": "Database: company_db\nSchema: employees(emp_id, name, age, department, salary)\nQuestion: Find the names of all employees in the Sales department who earn more than 6000.",
    },
    {
        "instruction": "You are a senior SQL developer. Write a SQL query that answers the user's question based on the database schema.",
        "input": "Database: retail_store\nSchema: orders(order_id, customer_id, total_amount, order_date)\nQuestion: What is the total amount spent by each customer? Sort the results by the total amount in descending order.",
    },
    {
        "instruction": "You are a senior SQL developer. Write a SQL query that answers the user's question based on the database schema.",
        "input": "Database: school_sys\nSchema: students(student_id, first_name, major_id), majors(major_id, major_name)\nQuestion: Show the first names of all students along with their major names.",
    },
    {
        "instruction": "You are a helpful AI assistant.",
        "input": "Hello, can you briefly explain the difference between a database and a spreadsheet?",
    },
]



In [61]:
# 仅测试用例 1
case = test_cases[0]
prompt = prompt.format(case["instruction"], case["input"], "")
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

# 记录输入 Prompt 的 token 长度
prompt_len = inputs["input_ids"].shape[1]

# 执行生成
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.7,  # 略微增加一点随机性，避免贪婪搜索直接命中 EOS
    top_p=0.9,
)

# 1. 核心：只截取模型新吐出来的 Token
new_tokens = outputs[0][prompt_len:]

# 2. 诊断输出
print(f"生成的 Token 数量: {len(new_tokens)}")
print(f"生成的 Token ID 列表: {new_tokens.tolist()[:10]}")
print(f"直接解码新 Token 的内容: {repr(tokenizer.decode(new_tokens))}")

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


生成的 Token 数量: 22
生成的 Token ID 列表: [4963, 836, 4393, 8420, 5401, 9476, 220, 284, 220, 364]
直接解码新 Token 的内容: "SELECT name FROM employees WHERE department  =  'Sales' AND salary  >  6000\n<|eot_id|>"


In [55]:
# 打印数据集前 3 条训练文本的末尾部分
for i in range(3):
    sample_text = dataset[i]["text"]
    print(f"--- 样本 {i} ---")
    print(
        sample_text[-100:]
    )  # 确保 Response 后面真的有 SQL 语句，而不是直接接 eos_token

--- 样本 0 ---
 departments are older than 56 ?
### Response:
SELECT count(*) FROM head WHERE age  >  56
<|eot_id|>
--- 样本 1 ---
nts ordered by age.
### Response:
SELECT name ,  born_state ,  age FROM head ORDER BY age
<|eot_id|>
--- 样本 2 ---
h department.
### Response:
SELECT creation ,  name ,  budget_in_billions FROM department
<|eot_id|>


In [62]:
# 遍历测试并生成结果
for i, case in enumerate(test_cases):
    print(f"\n--- 测试用例 {i+1} ---")
    # 将输入格式化为我们训练时使用的模板
    input_prompt = prompt_template.format(
        case["instruction"], case["input"], ""  # 输出部分留空，让模型自己生成
    )

    # 转换为模型可读的 tensor
    inputs = tokenizer([input_prompt], return_tensors="pt").to("cuda")

    # 生成回答 (max_new_tokens 限制输出长度，防止无限生成)
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)

    # 解码并提取模型生成的文本
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # 截取 "### Response:" 之后的内容
    final_output = response.split("### Response:\n")[-1].strip()

    print(
        f"输入问题: {case['input'].split('Question: ')[-1] if 'Question: ' in case['input'] else case['input']}"
    )
    print(f"模型输出: {final_output}")

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- 测试用例 1 ---


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


输入问题: Find the names of all employees in the Sales department who earn more than 6000.
模型输出: SELECT name FROM employees WHERE department  =  'Sales' AND salary  >  6000

--- 测试用例 2 ---


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


输入问题: What is the total amount spent by each customer? Sort the results by the total amount in descending order.
模型输出: SELECT customer_id,  sum(total_amount) FROM orders GROUP BY customer_id ORDER BY sum(total_amount) DESC

--- 测试用例 3 ---


Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


输入问题: Show the first names of all students along with their major names.
模型输出: SELECT T1.first_name,  T2.major_name FROM students AS T1 JOIN majors AS T2 ON T1.major_id  =  T2.major_id

--- 测试用例 4 ---
输入问题: Hello, can you briefly explain the difference between a database and a spreadsheet?
模型输出: Yes, of course! A database is a structured collection of data that is typically used to support specific queries, analyses, and applications. It is designed to efficiently manage large amounts of data and provide a way to retrieve specific data quickly. On the other hand, a spreadsheet is an electronic document that is used to store, organize, and manipulate data. It is designed to provide a flexible and user-friendly way to work with data, but it is not typically used to manage large amounts of data or support complex queries.
